In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from spatial import find_municipality_udf


In [0]:
bronze_tomtom_incidents = spark.readStream.table("bg_traffic.bg_traffic_bronze.tomtom_incident")


In [0]:
tomtom_incidents_checkpoint = "/Volumes/bg_traffic/bg_traffic_silver/checkpoints/tomtom_incident"

In [0]:
bronze_tomtom_incidents.printSchema()

### Flattening

In [0]:
tomtom_incidents = bronze_tomtom_incidents.select(
    F.explode("data.incidents").alias("incident"),
    "fetched_at",
    "_ingestion_timestamp",
    "_source_file"
)


In [0]:
tomtom_incidents_flatten = tomtom_incidents.select(
    F.col("incident.geometry.coordinates").alias("coordinates"),
    F.col("fetched_at"),
    F.col("incident.geometry.type").alias("geometry_type"),
    F.col("incident.properties.iconCategory").alias("icon_category"),
    F.col("_ingestion_timestamp"),
    F.col("_source_file")
)



### Casting

In [0]:
tomtom_incidents_types = tomtom_incidents_flatten\
    .withColumn("fetched_at", F.to_timestamp("fetched_at"))\
    .withColumn("icon_category", F.col("icon_category").cast("integer"))\
    .withColumn("record_id", F.md5(
        F.concat_ws(
            "||",
            F.coalesce(F.col("coordinates").cast("string"), F.lit("")),
            F.coalesce(F.col("icon_category").cast("string"), F.lit("")),
            F.coalesce(F.col("fetched_at").cast("string"), F.lit(""))
        )
    ))\
    .withColumn("start_longitude", F.col("coordinates")[0][0].cast("double"))\
    .withColumn("start_latitude", F.col("coordinates")[0][1].cast("double"))\
    .withColumn("end_longitude", F.element_at(F.col("coordinates"),-1)[0].cast("double"))\
    .withColumn("end_latitude", F.element_at(F.col("coordinates"),-1)[1].cast("double"))\
    .withColumn("coordinates", F.expr("transform(coordinates, c -> struct(c[1] as latitude, c[0] as longitude))"))



In [0]:
tomtom_incidents_types.printSchema()

### Valid

In [0]:
tomtom_incidents_valid = tomtom_incidents_types.filter(
    (F.col("record_id").isNotNull()) &
    (F.col("start_latitude").between(44.0, 45.5)) &
    (F.col("end_latitude").between(44.0, 45.5)) &
    (F.col("start_longitude").between(19.5, 21.0)) &
    (F.col("end_longitude").between(19.5, 21.0)) &
    (F.col("coordinates").isNotNull()) &
    (F.size(F.col("coordinates")) > 0)
).withColumn("municipality_name", find_municipality_udf(F.col("start_latitude"), F.col("start_longitude")))



### Merge

In [0]:
def merge_tomtom_incidents(df_source, batch_id):
    if df_source.isEmpty():
        return
    
    tomtom_incidents_window = Window.partitionBy("record_id")\
    .orderBy(F.col("fetched_at").desc())

    tomtom_incidents_dedup = df_source\
    .withColumn("row_num", F.row_number().over(tomtom_incidents_window))\
        .filter(F.col("row_num") == 1)\
            .drop("row_num")

    
    silver_table_tomtom_incidents = DeltaTable.forName(
        spark,
        "bg_traffic.bg_traffic_silver.tomtom_incident"
    ).alias("target").merge(tomtom_incidents_dedup.alias("source"),"target.record_id = source.record_id")\
        .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
                .execute()
            



In [0]:
query = (
    tomtom_incidents_valid
    .writeStream
    .foreachBatch(merge_tomtom_incidents)
    .option("checkpointLocation", tomtom_incidents_checkpoint)
    .trigger(availableNow=True)
    .start()
)

In [0]:
tomtom_incidents_valid.printSchema()

In [0]:
%sql

SELECT * FROM bg_traffic.bg_traffic_silver.tomtom_incident